# Análise de dependencia da rede neural

## Coleta e Limpeza dos Dados

In [1]:
import numpy as np
import machinelearning_mnpkllps.Preprocessing as PP
import machinelearning_mnpkllps.Resampling as RA
import machinelearning_mnpkllps.NeuralNetwork as NN
import machinelearning_mnpkllps.Metrics as MT

In [2]:
# Leitura dos dados
dados = np.genfromtxt('../data/RESULTADOS_2025.csv', 
                      delimiter=';', 
                      skip_header=1, 
                      encoding='latin-1', 
                      usecols=(12, 22, 23, 24, 25, 30, 36, 37, 38, 39, 40))

# Colunas selecionadas:
# UF, NOTA_CN, NOTA_CH, NOTA_LC, NOTA_MT, LINGUA, NOTA_R1, NOTA_R2, NOTA_R3, NOTA_R4, NOTA_R5

In [3]:
# Leitura dos dados
R_total = np.genfromtxt('../data/RESULTADOS_2025.csv', 
                      delimiter=';', 
                      skip_header=1, 
                      encoding='latin-1', 
                      usecols=(41))

# Colunas selecionadas:
# NOTA_RT

In [4]:
data = dados.copy()
rng = np.random.default_rng(450)

linhas_com_nan = np.any(np.isnan(data), axis=1)

linhas_validas = ~linhas_com_nan

n = int(3e5)
clean_data = data[linhas_validas]

_, clean_data, _, _ = PP.train_test_split(clean_data, clean_data, test_size=0.10)




print(f"Número de linhas originais: {data.shape[0]}")
print(f"Número de linhas da base limpa: {clean_data.shape[0]}")

Número de linhas originais: 4810772
Número de linhas da base limpa: 324435


In [5]:
clean_data = PP.ColumnCategoricalEncoder([0], drop_first=True).fit_transform(clean_data)

In [6]:
col = {
    'UF': 0,
    'NOTA_CN': 26, 
    'NOTA_CH': 27, 
    'NOTA_LC': 28, 
    'NOTA_MT': 29, 
    'LINGUA': 30, 
    'NOTA_R1': 31, 
    'NOTA_R2': 32, 
    'NOTA_R3': 33, 
    'NOTA_R4': 34, 
    'NOTA_R5': 35
}

In [7]:
X = clean_data[:, :col['LINGUA']+1]
y = clean_data[:, col['NOTA_R1']:]

display(X)
display(y)

array([[  0. ,   0. ,   0. , ..., 554.7, 380.3,   0. ],
       [  0. ,   0. ,   0. , ..., 523.3, 543.7,   0. ],
       [  0. ,   0. ,   0. , ..., 610.5, 619.3,   1. ],
       ...,
       [  0. ,   0. ,   0. , ..., 443.3, 412.1,   1. ],
       [  0. ,   0. ,   0. , ..., 438.8, 331.9,   0. ],
       [  0. ,   0. ,   0. , ..., 347.4, 381.8,   1. ]],
      shape=(324435, 31))

array([[100., 120., 120., 100., 120.],
       [160., 160., 160., 160., 160.],
       [140., 160., 120., 140., 180.],
       ...,
       [120., 160., 140., 180., 140.],
       [ 80., 120., 100.,  80.,  80.],
       [100., 120., 120., 120., 120.]], shape=(324435, 5))

In [8]:
x_train, x_aux, y_train, y_aux = PP.train_test_split(X, y, test_size=0.40)
x_valid, x_test, y_valid, y_test = PP.train_test_split(x_aux, y_aux, test_size=0.50)

display(x_train.shape, x_test.shape)
display(x_valid.shape, x_valid.shape)
display(y_test.shape, y_test.shape)

(194661, 31)

(64887, 31)

(64887, 31)

(64887, 31)

(64887, 5)

(64887, 5)

In [9]:
x_train = PP.MinMaxScaler().fit_transform(x_train)
x_valid = PP.MinMaxScaler().fit_transform(x_valid)
x_test = PP.MinMaxScaler().fit_transform(x_test)

y_train = PP.MinMaxScaler().fit_transform(y_train)
y_valid = PP.MinMaxScaler().fit_transform(y_valid)
y_test = PP.MinMaxScaler().fit_transform(y_test)

## Realizando a regração separadamente para cada variavel

In [10]:
NNs = []
for i in range(5):
    nn = NN.NeuralNetwork([31, 64, 32, 16, 1], 'ReLU', 'sigmoid', bias_interval=(0, 0), weight_interval=(-1, 1), seed=782+i)
    nn.fit(x_train, y_train[:, i].reshape(-1, 1), learn_rate=0.001, batch=32, epochs=200, debug = True, tol=1e-8)

    NNs.append(nn)

Iniciando epoca 0, com erro 0.36352128893350355
Iniciando epoca 1, com erro 0.34973968767764013
Iniciando epoca 2, com erro 0.1288869012756618
Iniciando epoca 3, com erro 0.08709303851893273
Iniciando epoca 4, com erro 0.07038881818266231
Iniciando epoca 5, com erro 0.051278576031660326
Iniciando epoca 6, com erro 0.04137024578365206
Iniciando epoca 7, com erro 0.03560714670351842
Iniciando epoca 8, com erro 0.033138155763387765
Iniciando epoca 9, com erro 0.031659121860508406
Iniciando epoca 10, com erro 0.0308940829859556
Iniciando epoca 11, com erro 0.03021949406072037
Iniciando epoca 12, com erro 0.0298581156842267
Iniciando epoca 13, com erro 0.02944607143587682
Iniciando epoca 14, com erro 0.02910283350135604
Iniciando epoca 15, com erro 0.02889066034287321
Iniciando epoca 16, com erro 0.02872688321475475
Iniciando epoca 17, com erro 0.028499657771048174
Iniciando epoca 18, com erro 0.028286375128333766
Iniciando epoca 19, com erro 0.028130246098359353
Iniciando epoca 20, com err

In [11]:
R1_train = y_train[:, 0].reshape(-1, 1)
R2_train = y_train[:, 1].reshape(-1, 1)
R3_train = y_train[:, 2].reshape(-1, 1)
R4_train = y_train[:, 3].reshape(-1, 1)
R5_train = y_train[:, 4].reshape(-1, 1)
RT_train = R1_train + R2_train + R3_train + R4_train + R5_train

R1_prev_train = NNs[0].prevision(x_train)
R2_prev_train = NNs[1].prevision(x_train)
R3_prev_train = NNs[2].prevision(x_train)
R4_prev_train = NNs[3].prevision(x_train)
R5_prev_train = NNs[4].prevision(x_train)
RT_prev_train = R1_prev_train + R2_prev_train + R3_prev_train + R4_prev_train + R5_prev_train

print(f'Erro treino Nota R1: {MT.RMSE(R1_prev_train, R1_train)*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R2_prev_train, R2_train)*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R3_prev_train, R3_train)*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R4_prev_train, R4_train)*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R5_prev_train, R5_train)*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_train, RT_train)*200}')

Erro treino Nota R1: 32.064264242459494
Erro treino Nota R2: 37.74424931845618
Erro treino Nota R3: 33.28889189760065
Erro treino Nota R4: 37.50696089152333
Erro treino Nota R5: 52.938702846514076
Erro treino Nota RT: 167.67269104021338


In [12]:
R1_valid = y_valid[:, 0].reshape(-1, 1)
R2_valid = y_valid[:, 1].reshape(-1, 1)
R3_valid = y_valid[:, 2].reshape(-1, 1)
R4_valid = y_valid[:, 3].reshape(-1, 1)
R5_valid = y_valid[:, 4].reshape(-1, 1)
RT_valid = R1_valid + R2_valid + R3_valid + R4_valid + R5_valid

R1_prev_valid = NNs[0].prevision(x_valid)
R2_prev_valid = NNs[1].prevision(x_valid)
R3_prev_valid = NNs[2].prevision(x_valid)
R4_prev_valid = NNs[3].prevision(x_valid)
R5_prev_valid = NNs[4].prevision(x_valid)
RT_prev_valid = R1_prev_valid + R2_prev_valid + R3_prev_valid + R4_prev_valid + R5_prev_valid

print(f'Erro treino Nota R1: {MT.RMSE(R1_prev_valid, R1_valid)*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R2_prev_valid, R2_valid)*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R3_prev_valid, R3_valid)*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R4_prev_valid, R4_valid)*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R5_prev_valid, R5_valid)*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_valid, RT_valid)*200}')

Erro treino Nota R1: 32.451315871419375
Erro treino Nota R2: 38.00476458343596
Erro treino Nota R3: 33.632425532504854
Erro treino Nota R4: 37.867447846248716
Erro treino Nota R5: 53.20090792499431
Erro treino Nota RT: 169.4382110016311


In [13]:
R1_test = y_test[:, 0].reshape(-1, 1)
R2_test = y_test[:, 1].reshape(-1, 1)
R3_test = y_test[:, 2].reshape(-1, 1)
R4_test = y_test[:, 3].reshape(-1, 1)
R5_test = y_test[:, 4].reshape(-1, 1)
RT_test = R1_test + R2_test + R3_test + R4_test + R5_test

R1_prev_test = NNs[0].prevision(x_test)
R2_prev_test = NNs[1].prevision(x_test)
R3_prev_test = NNs[2].prevision(x_test)
R4_prev_test = NNs[3].prevision(x_test)
R5_prev_test = NNs[4].prevision(x_test)
RT_prev_test = R1_prev_test + R2_prev_test + R3_prev_test + R4_prev_test + R5_prev_test

print(f'Erro treino Nota R1: {MT.RMSE(R1_prev_test, R1_test)*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R2_prev_test, R2_test)*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R3_prev_test, R3_test)*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R4_prev_test, R4_test)*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R5_prev_test, R5_test)*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_test, RT_test)*200}')


Erro treino Nota R1: 31.895523029964146
Erro treino Nota R2: 37.507473271148775
Erro treino Nota R3: 33.105715241806536
Erro treino Nota R4: 37.3203092620073
Erro treino Nota R5: 52.902295743381075
Erro treino Nota RT: 166.78304069908685


In [14]:
print("Nota real:")
print(RT_train.min(), RT_train.max(), RT_train.mean())

print("Nota prevista:")
print(RT_prev_train.min(), RT_prev_train.max(), RT_prev_train.mean())

Nota real:
0.0 5.0 2.9479608139278026
Nota prevista:
0.4000000443732165 4.6209493781806374 2.9661755518722104


## Realizando a regração para todas juntas

In [15]:
nn = NN.NeuralNetwork([31, 64, 32, 5], 'ReLU', 'sigmoid', bias_interval=(0, 0), weight_interval=(-1, 1), seed=147)
nn.fit(x_train, y_train, learn_rate=0.0005, batch=64, epochs=500, debug = True, tol=1e-8)

Iniciando epoca 0, com erro 0.9599068152593041
Iniciando epoca 1, com erro 0.7395744243577935
Iniciando epoca 2, com erro 0.6593829843303426
Iniciando epoca 3, com erro 0.6101670193304337
Iniciando epoca 4, com erro 0.5695487031628473
Iniciando epoca 5, com erro 0.5346807440475031
Iniciando epoca 6, com erro 0.5051329752162503
Iniciando epoca 7, com erro 0.4792472797563173
Iniciando epoca 8, com erro 0.45693300323438935
Iniciando epoca 9, com erro 0.4375813739953005
Iniciando epoca 10, com erro 0.4190009879637249
Iniciando epoca 11, com erro 0.39775029763432246
Iniciando epoca 12, com erro 0.38226299948473036
Iniciando epoca 13, com erro 0.3706464438306857
Iniciando epoca 14, com erro 0.35999202897470994
Iniciando epoca 15, com erro 0.34986798419686205
Iniciando epoca 16, com erro 0.33989559506938727
Iniciando epoca 17, com erro 0.3297327312034135
Iniciando epoca 18, com erro 0.31927603350710065
Iniciando epoca 19, com erro 0.30947265944370106
Iniciando epoca 20, com erro 0.30120346810

In [16]:
R_train = y_train
RT_train = np.sum(R_train, axis=1)

R_prev_train = nn.prevision(x_train)
RT_prev_train = np.sum(R_prev_train, axis=1)

print(f'Erro treino Nota R1: {MT.RMSE(R_prev_train[:, 0], R_train[:, 0])*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R_prev_train[:, 1], R_train[:, 1])*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R_prev_train[:, 2], R_train[:, 2])*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R_prev_train[:, 3], R_train[:, 3])*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R_prev_train[:, 4], R_train[:, 4])*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_train, RT_train)*200}')

Erro treino Nota R1: 32.004585621253106
Erro treino Nota R2: 37.76146242525672
Erro treino Nota R3: 33.15937198536653
Erro treino Nota R4: 37.496570643793426
Erro treino Nota R5: 52.91123386998431
Erro treino Nota RT: 167.90815830849417


In [17]:
R_valid = y_valid
RT_valid = np.sum(R_valid, axis=1)

R_prev_valid = nn.prevision(x_valid)
RT_prev_valid = np.sum(R_prev_valid, axis=1)

print(f'Erro treino Nota R1: {MT.RMSE(R_prev_valid[:, 0], R_valid[:, 0])*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R_prev_valid[:, 1], R_valid[:, 1])*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R_prev_valid[:, 2], R_valid[:, 2])*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R_prev_valid[:, 3], R_valid[:, 3])*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R_prev_valid[:, 4], R_valid[:, 4])*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_valid, RT_valid)*200}')

Erro treino Nota R1: 32.35292171623156
Erro treino Nota R2: 37.970833677098014
Erro treino Nota R3: 33.45449935223322
Erro treino Nota R4: 37.82354872237275
Erro treino Nota R5: 53.12916297788962
Erro treino Nota RT: 169.47209896573708


In [18]:
for i in range(5):
    baseline = np.full_like(y_valid[:, i], np.mean(y_train[:, i]))

    print(
        f"R{i+1}:",
        MT.RMSE(baseline, y_valid[:, i])
    )

R1: 0.18935145412629062
R2: 0.21809557599618598
R3: 0.19723869862102933
R4: 0.22560059034404858
R5: 0.31980988439124225
